> 📎 **Appendix notebook — reference style.** This is one of the optional appendices (see `README.md`). Unlike the main course notebooks, appendices are written as a demo / reference: they focus on *seeing* a library at work rather than on interactive exercises. You won't find the full Solution / Debug-me / Self-assessment scaffolding here — and if you don't have the optional dependency installed, the notebook will stop at the first import. That's by design.

---
# 📓 Notebook A1 (DS) — Classical Forecasting: ARIMA / SARIMA / ETS

> **Module:** Data Science · **Type:** Appendix · **Estimated time:** 75–90 min · **Difficulty:** Intermediate → Advanced

Notebook 14 introduced time-series forecasting via Holt–Winters. This appendix is the *deep dive*: stationarity, the Box–Jenkins family (AR, MA, ARIMA, SARIMA), and the modern state-space ETS framework. By the end you'll know how to:

- Diagnose a series (stationarity, seasonality, trend) and pick a model class.
- Read **ACF/PACF** plots like a craftsperson.
- Fit a `SARIMAX` model in `statsmodels` and produce **prediction intervals** that hold up to a stakeholder review.
- Compare classical models honestly using rolling-origin cross-validation.

---

## ✅ Prerequisites
- Notebook 14 (Time Series & Forecasting Basics).
- Notebook 13 (Statistics Basics) — confidence intervals, hypothesis tests.

## 📦 Install

```bash
pip install statsmodels      # bundled with the course requirements; no extra step usually needed
```


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
print("statsmodels", sm.__version__)


## 1. A flavoured business time series

Daily product searches with weekly seasonality, an upward trend, and a one-off promo bump. 18 months of data.


In [ ]:
rng = np.random.default_rng(0)
n = 540                                              # 18 months × 30 days
dates = pd.date_range("2024-01-01", periods=n, freq="D")
trend = 0.04 * np.arange(n)
weekly = 8 * np.sin(2 * np.pi * np.arange(n) / 7) + 4 * np.cos(2 * np.pi * np.arange(n) / 7)
yearly = 6 * np.sin(2 * np.pi * np.arange(n) / 365)
promo  = np.where((np.arange(n) > 200) & (np.arange(n) < 220), 12, 0)
noise  = rng.normal(0, 2.5, size=n)
y      = 50 + trend + weekly + yearly + promo + noise

ts = pd.Series(y, index=dates, name="searches")

fig, ax = plt.subplots(figsize=(11, 3.5))
ts.plot(ax=ax); ax.set_title("Daily product searches"); plt.show()
print(ts.describe().round(2))


## 2. Diagnose first — is it stationary?

A series is **stationary** if its statistical properties (mean, variance, autocorrelation) don't change over time. Classical models (ARMA, ARIMA) require stationarity *after differencing*.

Two tests, used together:

| Test | Null hypothesis | If p < 0.05 |
|---|---|---|
| **ADF** (Augmented Dickey-Fuller) | non-stationary (has unit root) | series IS stationary |
| **KPSS** | stationary | series is NOT stationary |

Conflicting verdicts (both reject, or both fail to reject) are normal — that's why we always use both.


In [ ]:
def stationarity_report(s, name="series"):
    adf_p  = adfuller(s.dropna(), autolag="AIC")[1]
    kpss_p = kpss(s.dropna(), regression="c", nlags="auto")[1]
    print(f"  {name:25s}  ADF p={adf_p:.4f}  KPSS p={kpss_p:.4f}  "
          f"→ {'stationary' if adf_p < 0.05 and kpss_p > 0.05 else 'NOT stationary'}")

stationarity_report(ts, "raw")
stationarity_report(ts.diff(),                       "1st diff")
stationarity_report(ts.diff().diff(7),               "1st diff + seasonal diff(7)")


**Interpretation.** The raw series fails ADF (has a trend). After **first difference + seasonal difference at lag 7**, both tests agree it's stationary. That's our `(d, D) = (1, 1)` for the SARIMA spec below.


## 3. Read the ACF/PACF — the Box–Jenkins shortcut

The **ACF** (auto-correlation function) shows how the series correlates with its own lagged values. The **PACF** (partial ACF) removes the effect of shorter lags. Together they tell you the AR and MA orders.

| Pattern | Likely model |
|---|---|
| ACF tails off, PACF cuts off after p | **AR(p)** |
| ACF cuts off after q, PACF tails off | **MA(q)** |
| Both tail off | **ARMA(p, q)** |


In [ ]:
diff = ts.diff().diff(7).dropna()
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.2))
plot_acf(diff, lags=30, ax=ax1); ax1.set_title("ACF of doubly-differenced series")
plot_pacf(diff, lags=30, ax=ax2, method="ywm"); ax2.set_title("PACF")
plt.tight_layout(); plt.show()


From the plots we see a spike at lag 7 (the seasonal effect) on both ACF and PACF — classic SARIMA territory. A reasonable first try: **SARIMA(1, 1, 1) × (1, 1, 1, 7)**.

In the notation `SARIMA(p, d, q) × (P, D, Q, m)`:
- `p, d, q` — non-seasonal AR order, differencing order, MA order
- `P, D, Q` — same for seasonal
- `m` — seasonal period (7 for weekly with daily data; 12 for monthly with monthly data; 24 for hourly daily-cycle)


## 4. Fit SARIMA with `statsmodels`


In [ ]:
train = ts[:-30]; test = ts[-30:]

model = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7),
                enforce_stationarity=False, enforce_invertibility=False)
res = model.fit(disp=False)
print(res.summary().tables[1])


In [ ]:
# Forecast 30 days with 95% intervals
fc = res.get_forecast(steps=len(test))
mean = fc.predicted_mean
ci   = fc.conf_int(alpha=0.05)

fig, ax = plt.subplots(figsize=(11, 3.8))
train[-90:].plot(ax=ax, label="train (last 90 days)")
test.plot(ax=ax, label="actual", color="black")
mean.plot(ax=ax, label="forecast", color="C2")
ax.fill_between(mean.index, ci.iloc[:, 0], ci.iloc[:, 1], color="C2", alpha=0.2,
                label="95% PI")
ax.legend(); ax.set_title("SARIMA forecast — 30 days out"); plt.show()

from sklearn.metrics import mean_absolute_error, mean_squared_error
mae  = mean_absolute_error(test, mean)
rmse = np.sqrt(mean_squared_error(test, mean))
print(f"MAE  = {mae:.2f}     RMSE = {rmse:.2f}")


## 5. Residual diagnostics — did we actually fit the noise?

A correctly-specified model leaves *no signal* in the residuals. The Ljung–Box test should NOT reject.


In [ ]:
fig = res.plot_diagnostics(figsize=(11, 6)); plt.tight_layout(); plt.show()

from statsmodels.stats.diagnostic import acorr_ljungbox
lb = acorr_ljungbox(res.resid, lags=[10, 20], return_df=True)
print("Ljung-Box (want p > 0.05):")
print(lb.round(4))


## 6. ETS — exponential smoothing, modern state-space form

ETS = **Error**, **Trend**, **Seasonality**. Each component can be additive or multiplicative. Often *as accurate as* SARIMA, with fewer knobs.

| Component | Choices |
|---|---|
| Error (`E`) | additive (`A`), multiplicative (`M`) |
| Trend (`T`) | none (`N`), additive (`A`), damped additive (`Ad`) |
| Seasonality (`S`) | none (`N`), additive (`A`), multiplicative (`M`) |


In [ ]:
ets = ExponentialSmoothing(train, trend="add", seasonal="add", seasonal_periods=7)
fit = ets.fit(optimized=True, use_brute=True)
fc  = fit.forecast(steps=len(test))

# Bootstrap-ish prediction intervals via residual std
sigma = fit.resid.std()
lo, hi = fc - 1.96 * sigma, fc + 1.96 * sigma

fig, ax = plt.subplots(figsize=(11, 3.8))
train[-90:].plot(ax=ax, label="train")
test.plot(ax=ax, color="black", label="actual")
fc.plot(ax=ax, color="C3", label="ETS forecast")
ax.fill_between(fc.index, lo, hi, color="C3", alpha=0.2, label="95% PI")
ax.legend(); ax.set_title("ETS(A, A, A) forecast"); plt.show()

print(f"ETS  MAE = {mean_absolute_error(test, fc):.2f}   "
      f"RMSE = {np.sqrt(mean_squared_error(test, fc)):.2f}")


## 7. Honest comparison — rolling-origin cross-validation

A single train/test split is a brittle benchmark — your forecast quality can vary by an order of magnitude across cuts of the data. The professional answer is **rolling-origin** (a.k.a. *time-series cross-validation*).


In [ ]:
def rolling_eval(series, train_fn, horizon=14, n_splits=8, step=30):
    n = len(series)
    maes = []
    for k in range(n_splits):
        end_train = n - horizon - k * step
        if end_train < 60: break
        tr, te = series[:end_train], series[end_train:end_train + horizon]
        try:
            fc = train_fn(tr, horizon)
            maes.append(mean_absolute_error(te, fc))
        except Exception as e:
            maes.append(np.nan)
    return np.array(maes)

def fit_sarima(tr, h):
    m = SARIMAX(tr, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7),
                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
    return m.forecast(steps=h)

def fit_ets(tr, h):
    return ExponentialSmoothing(tr, trend="add", seasonal="add", seasonal_periods=7
            ).fit(optimized=True).forecast(steps=h)

def fit_naive(tr, h):     # seasonal naive — last week's pattern
    return pd.Series(tr.values[-7:].tolist() * (h // 7 + 1)[:h],
                     index=pd.date_range(tr.index[-1] + pd.Timedelta(days=1), periods=h))

mae_sar = rolling_eval(ts, fit_sarima)
mae_ets = rolling_eval(ts, fit_ets)
print(f"SARIMA  MAE = {np.nanmean(mae_sar):.2f} ± {np.nanstd(mae_sar):.2f}")
print(f"ETS     MAE = {np.nanmean(mae_ets):.2f} ± {np.nanstd(mae_ets):.2f}")


**Reading the result.** Look at *both* the mean AND the standard deviation. A model that wins on average but is unstable across cuts is risky — and *that* is what rolling-origin reveals.


## 8. Decision rubric — pick a classical model

```
1. Plot the series.  Trend?  Seasonality?  Outliers?
2. Test for stationarity (ADF + KPSS).  Difference until stationary.
3. Read ACF/PACF on the differenced series to guess (p, q).
4. Fit SARIMA(p, d, q)(P, D, Q, m) AND ETS — they have different strengths.
5. Evaluate with rolling-origin CV.  Pick the lower-mean, lower-variance one.
6. Check residuals (Ljung-Box).  If still autocorrelated, raise (p) or (P).
```

When SARIMA usually wins: strong autocorrelation, complex seasonality, need explainable coefficients.
When ETS usually wins: smooth, dominantly trend-driven series; fewer parameters to tune.


## 🧪 Exercises

### Exercise 1 — Add a regressor (`SARIMAX`'s "X")
Add a binary `promo` indicator series and fit a `SARIMAX` with `exog=promo`. Does the AIC drop? Are the residual ACFs cleaner?


In [ ]:
# ── Exercise 1 solution ──────────────────────────────────────────────────
promo_indicator = pd.Series(np.where((np.arange(n) > 200) & (np.arange(n) < 220), 1, 0),
                            index=dates)

train_x, test_x = promo_indicator[:-30], promo_indicator[-30:]
model_x = SARIMAX(train, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7),
                  exog=train_x, enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
print(f"SARIMA   AIC = {res.aic:.1f}")
print(f"SARIMAX  AIC = {model_x.aic:.1f}  ← lower is better")
fc_x = model_x.get_forecast(steps=len(test), exog=test_x.values.reshape(-1, 1)).predicted_mean
print(f"MAE without exog: {mean_absolute_error(test, mean):.2f}")
print(f"MAE with exog:    {mean_absolute_error(test, fc_x):.2f}")


### Exercise 2 — Auto-search for the best `(p, d, q)`
Without using `pmdarima`, write a small grid search over `p, q ∈ {0,1,2}` and `P, Q ∈ {0,1}` that picks the SARIMA with the lowest **AIC**. Report which order won and re-run the rolling-origin evaluation on it.


In [ ]:
# ── Exercise 2 solution ──────────────────────────────────────────────────
best, best_aic = None, np.inf
for p in range(3):
    for q in range(3):
        for P in range(2):
            for Q in range(2):
                try:
                    m = SARIMAX(train, order=(p, 1, q), seasonal_order=(P, 1, Q, 7),
                                enforce_stationarity=False, enforce_invertibility=False).fit(disp=False)
                    if m.aic < best_aic:
                        best, best_aic = (p, 1, q, P, 1, Q, 7), m.aic
                except Exception:
                    pass
print(f"best order: SARIMA{best[:3]} x {best[3:]}  AIC={best_aic:.1f}")
def fit_best(tr, h):
    p, d, q, P, D, Q, m = best
    return SARIMAX(tr, order=(p, d, q), seasonal_order=(P, D, Q, m),
                   enforce_stationarity=False, enforce_invertibility=False
                   ).fit(disp=False).forecast(steps=h)
mae_best = rolling_eval(ts, fit_best)
print(f"best-grid  MAE = {np.nanmean(mae_best):.2f} ± {np.nanstd(mae_best):.2f}")


## 🧠 Key takeaways

- Diagnose **before** modelling. ADF + KPSS for stationarity; ACF + PACF for orders.
- `SARIMAX(p, d, q)(P, D, Q, m)` and `ExponentialSmoothing(E, T, S)` cover 90 % of classical use cases.
- Use **rolling-origin CV** to compare honestly — a single split overestimates by a lot.
- Always check **residuals**. A clean Ljung-Box test is a non-negotiable sanity check.

## 🚀 Next step

[`A2_forecasting_prophet_libraries.ipynb`](./A2_forecasting_prophet_libraries.ipynb) — Prophet, NeuralProphet, sktime, Darts.
